In [1]:
from langchain_ollama import OllamaEmbeddings
# from sklearn.metrics.pairwise import cosine_similarity
# from scipy.spatial.distance import cosine

import numpy as np
import pandas as pd

from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans



In [2]:

embedder = OllamaEmbeddings(model="nomic-embed-text")


In [ ]:
# sacar el texto
df = pd.read_csv("../data/description_framework_table.csv")
df.head()

# lista de columna "description"
queries = df["Description"].tolist()
# print(queries)

In [4]:
# embed
vectors = [embedder.embed_query(x) for x in queries]
# print(vectors)

In [16]:
# queries_len = [len(x) for x in queries]
# vectors_len = [len(x) for x in vectors]
# print(f"queries length: {queries_len}\nvectors length: {vectors_len}")

To get the most similar queries to group them for retrieval, i will use k-means to get clusters of the most similar embedding vectors of the queries.

The retrieval method used in the app (similarity_seach()) uses cosine similarity. K-means, on the other hand, uses euclidean distance to compute the clustering, which means that the magnitude of vectors is important using this method, making it different from cosine similarity. To make these two methods comparable, it is necessary to normalize the vectors before applying k-means, so that the euclidean distance results are comparable to cosine similarity.

In [18]:
# normalize vectors
normalized_embeddings = normalize(vectors, norm='l2')

# clustering
n_clusters = 5 # TODO: can be adjusted (elbow method)

kmeans = KMeans(n_clusters=n_clusters, random_state=0)
labels = kmeans.fit_predict(normalized_embeddings)
print(f"Labels: {labels}")

Labels: [0 4 4 2 2 0 0 0 1 4 0 0 1 4 2 2 1 0 3 1 3]


In [17]:
# Group queries by their assigned cluster label
clustered_queries = {}
for idx, label in enumerate(labels):
    clustered_queries.setdefault(label, []).append(queries[idx])

# Display grouped queries
for cluster_id, texts in clustered_queries.items():
    print(f"Cluster {cluster_id}:")
    for text in texts:
        print(f"  - {text}")
    print()

Cluster 0:
  - Describes the SUS, i.e., the PT, of the system of interest.
  - Describes the services, such as optimization, task planning, and visualization, which the DT provides to the users and the physical system.
  - Describes the time-scale use and the time rates for the DT services and DT-to-PT synchronization.
  - Describes the multiplicities, i.e., the internal twins that compose the DT system, which can be implemented in a centralized or decentralized way.
  - Describes the tools or enablers that are used to achieve the goals of the DT, i.e., they enable the DT to provide the DT services.
  - Describes the orchestration of the DT system, components, and services as a whole.
  - Describes the information exchange with external information systems not limited to other DTs.

Cluster 4:
  - Describes the available acting components in the DT constellation, i.e., the mechanisms the DT can use to act on the PT.
  - Describes the available sensing components in the DT constellation

The result is very satisfying, I'm going to split the first cluster into two to make it as granular as posible

In [14]:
# from sklearn.decomposition import PCA

# import matplotlib.pyplot as plt

# # reduce embeddings to 2D using PCA for visualizatio
# pca = PCA(n_components=2)
# reduced_embeddings = pca.fit_transform(normalized_embeddings)

# # plot
# plt.figure(figsize=(8, 6))
# scatter = plt.scatter(
#     reduced_embeddings[:, 0],
#     reduced_embeddings[:, 1],
#     c=labels,
#     cmap='tab10',
#     s=60,
#     edgecolor='k'
# )
# plt.xlabel('PCA Component 1')
# plt.ylabel('PCA Component 2')
# plt.title('KMeans Clusters of Query Embeddings')
# plt.colorbar(scatter, ticks=range(n_clusters), label='Cluster Label')
# plt.show()

In [15]:
# # calculate inertia for a range of cluster numbers
# inertia = []
# cluster_range = range(1, 11)
# for k in cluster_range:
#     kmeans_tmp = KMeans(n_clusters=k, random_state=0)
#     kmeans_tmp.fit(normalized_embeddings)
#     inertia.append(kmeans_tmp.inertia_)

# # plot the elbow curve
# plt.figure(figsize=(8, 4))
# plt.plot(cluster_range, inertia, marker='o')
# plt.xlabel('Number of clusters')
# plt.ylabel('Inertia')
# plt.title('Elbow Method For Optimal k')
# plt.xticks(cluster_range)
# plt.show()